# DriftEnv — GRPO Training Run

**Project:** DriftEnv — RL environment testing AI agent robustness under ambiguity and context shift  
**Live Space:** https://huggingface.co/spaces/harims95/driftenv  
**Repo:** https://github.com/harims95/driftenv  
**Training date:** <!-- FILL IN: e.g. April 26, 2026 -->  
**GPU:** A10G (switch from T4 after dry run confirms no errors)  
**Model:** Qwen2.5-1.5B-Instruct via Unsloth 4-bit  

---

**What this notebook does:**
1. Loads Qwen2.5-1.5B-Instruct with Unsloth (4-bit, LoRA)
2. Trains with GRPO against the live DriftEnv Space (20 training scenarios)
3. Evaluates untrained vs trained on 5 holdout scenarios
4. Saves reward curves and before/after bar chart to `assets/`

**Run order:** cells top-to-bottom. Do NOT skip Cell 2 (installs).

In [ ]:
# Run once per Colab session — kernel reset wipes installed packages
!pip install -q unsloth trl peft accelerate datasets
!pip install -q httpx pydantic huggingface_hub

# Verify key installs
import unsloth, trl
print(f"unsloth: {unsloth.__version__}  |  trl: {trl.__version__}")

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from huggingface_hub import login
from datasets import Dataset
import httpx, json, os

# Authenticate — paste your HF token when prompted
# Token needs: read + Inference API access
login()

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME    = "unsloth/Qwen2.5-1.5B-Instruct"  # Unsloth pre-optimized
MAX_SEQ_LEN   = 2048
LORA_RANK     = 16
DRIFTENV_URL  = "https://harims95-driftenv.hf.space"
HOLDOUT_ONLY  = False  # False during training; flip to True for eval runs

# Pre-warm the Space (first cold-start request can take ~30s)
import httpx as _hx
try:
    r = _hx.get(DRIFTENV_URL, timeout=60)
    print("Space status:", r.json().get("status"))
except Exception as e:
    print("Space warm-up failed — check URL:", e)

## Load Model with Unsloth

Uses 4-bit quantization to fit in 24 GB VRAM (A10G).  
**Critical: do NOT replace with plain `transformers.AutoModelForCausalLM`.**  
Unsloth's `FastLanguageModel` provides ~2x faster rollout generation —  
rollouts dominate RL runtime, so this is not optional.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimised checkpointing
)

# Confirm trainable params — should be ~10-15M for 1.5B + rank-16 LoRA
model.print_trainable_parameters()

## DriftEnv Client Wrapper

Talks to the live HF Space over HTTP.  
If the Space is cold, the first request takes ~30s — Cell 4 pre-warms it.  
All calls are synchronous (httpx blocking) — fine for sequential rollouts.

In [ ]:
def reset_env(task: str = "medium", holdout_only: bool = HOLDOUT_ONLY) -> dict:
    r = httpx.post(
        f"{DRIFTENV_URL}/reset",
        json={"task": task, "holdout_only": holdout_only},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

def step_env(action_text: str) -> dict:
    r = httpx.post(
        f"{DRIFTENV_URL}/step",
        json={"action": {"response": action_text}},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

# Smoke test
obs = reset_env(task="easy", holdout_only=False)
print("reset ok — instruction:", obs["observation"]["instruction"])

## Rollout Function (GRPO Reward Function)

This is the bridge between GRPO and DriftEnv.  
For each completion, it calls `/step`, gets the 4-component reward,  
and returns the weighted scalar to the trainer.

**Critical:** call `FastLanguageModel.for_inference(model)` before generating  
rollouts — this activates Unsloth's 2x inference kernel. Easy to forget.  
After training resumes, call `FastLanguageModel.for_training(model)`.

The `reward_log` global accumulates per-step component values for plotting.

In [ ]:
# Module-level logs — populated during training, consumed by plotting cells
_reward_log = []  # list of {step, R_format, R_interpretation, R_pivot, R_no_stale, total}
_error_log  = []  # list of {step, prompt_snippet, error}


def driftenv_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    GRPO reward function. Called once per rollout batch by GRPOTrainer.

    prompts:     list[str] -- one prompt per rollout (from the dataset)
    completions: list[str] -- one model completion per rollout
    returns:     list[float] -- scalar reward per rollout

    Uses easy (1-step) episodes: reset -> step -> reward.
    On any HTTP/timeout error, returns 0.0 for that rollout and logs to _error_log.
    Does NOT crash the training loop.

    NOTE: FastLanguageModel.for_inference(model) is NOT called here.
    This function only calls the DriftEnv HTTP API -- it does not generate
    with the model. for_inference belongs in evaluate_model (Cell 20).
    """
    rewards = []

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        step_idx = len(_reward_log)
        try:
            # Fresh 1-step episode for each rollout
            reset_env(task="easy", holdout_only=False)
            result = step_env(action_text=completion)

            # Unpack 4-component reward from info dict
            components = result.get("info", {}).get("rewards", {})
            r_fmt    = float(components.get("R_format",         0.0))
            r_interp = float(components.get("R_interpretation", 0.0))
            r_pivot  = float(components.get("R_pivot",          0.0))
            r_stale  = float(components.get("R_no_stale",       0.0))

            # Weighted sum -- mirrors server/app.py _compute_reward weights
            total = round(0.1*r_fmt + 0.3*r_interp + 0.4*r_pivot + 0.2*r_stale, 4)

            _reward_log.append({
                "step":             step_idx,
                "R_format":         r_fmt,
                "R_interpretation": r_interp,
                "R_pivot":          r_pivot,
                "R_no_stale":       r_stale,
                "total":            total,
            })
            rewards.append(total)

        except Exception as e:
            msg = f"rollout {i} | step={step_idx} | {type(e).__name__}: {e}"
            print(f"[reward_fn ERROR] {msg}")
            _error_log.append({"step": step_idx, "prompt_snippet": prompt[:80], "error": str(e)})
            rewards.append(0.0)

    if _error_log:
        print(f"[reward_fn] {len(_error_log)} errors total -- inspect _error_log for details")

    return rewards


## GRPO Config

150 training steps, batch 4, 4 generations per prompt.  
Total rollouts per step = `per_device_train_batch_size × num_generations` = 16.  

**For T4 dry run:** set `max_steps=3` to verify pipeline without burning credits.  
**For A10G real run:** use `max_steps=150` as configured below.

In [ ]:
config = GRPOConfig(
    output_dir="driftenv_grpo_run",
    max_steps=150,                      # set to 3 for T4 dry run
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=4,                  # GRPO group size — 4 completions compared per prompt
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    max_prompt_length=512,
    max_completion_length=256,
    temperature=0.9,
    report_to="none",                   # manual plotting below
)

print("Config ready. max_steps:", config.max_steps)
print("Rollouts per step:", config.per_device_train_batch_size * config.num_generations)

## Build Prompt Dataset

Fetch the 20 training scenarios from the live Space (holdout excluded).  
Format each as a prompt string matching the system prompt in `inference.py`.  
Wrap in a HuggingFace `Dataset` object for the trainer.

In [ ]:
import json as _json
import os as _os

# Populated by build_prompt_dataset -- used by evaluate_model in Cell 20
# Maps initial_instruction -> scenario_id for all 25 scenarios
_SCENARIO_LOOKUP = {}

PROMPT_TEMPLATE = (
    "You are an AI assistant working on real ML development tasks.\n"
    "Instructions may be vague, and requirements may shift mid-task.\n"
    "Respond concisely with your interpretation or pivot.\n\n"
    "Instruction: {instruction}\n\n"
    "Your response:"
)


def build_prompt_dataset(dataset_path: str = None) -> Dataset:
    """
    Load 20 training scenarios from dataset.json (holdout=False),
    format each as a prompt, return a HuggingFace Dataset.

    Also populates _SCENARIO_LOOKUP (instruction -> scenario_id) for Cell 20.
    dataset_path is auto-detected from common Colab repo locations if None.
    """
    global _SCENARIO_LOOKUP

    if dataset_path is None:
        candidates = [
            "server/dataset.json",
            "../server/dataset.json",
            "/content/driftenv/server/dataset.json",
            "/kaggle/working/driftenv/server/dataset.json",
            "/kaggle/input/driftenv/server/dataset.json",
        ]
        for path in candidates:
            if _os.path.exists(path):
                dataset_path = path
                break
        if dataset_path is None:
            raise FileNotFoundError(
                f"dataset.json not found. Tried: {candidates}\n"
                "Fix: !git clone https://github.com/harims95/driftenv && %cd driftenv"
            )

    with open(dataset_path, "r") as f:
        all_scenarios = _json.load(f)

    # Build lookup for Cell 20 (covers all 25 scenarios including holdout)
    _SCENARIO_LOOKUP = {s["initial_instruction"]: s["id"] for s in all_scenarios}
    print(f"Scenario lookup built: {len(_SCENARIO_LOOKUP)} entries")

    training = [s for s in all_scenarios if not s.get("holdout", False)]
    if len(training) != 20:
        raise AssertionError(
            f"Expected 20 training scenarios, got {len(training)}. "
            "Check holdout flags in server/dataset.json."
        )

    rows = [
        {
            "prompt":      PROMPT_TEMPLATE.format(instruction=s["initial_instruction"]),
            "scenario_id": s["id"],
        }
        for s in training
    ]

    dataset = Dataset.from_list(rows)
    print(f"Dataset ready: {len(dataset)} training prompts (holdout excluded)")
    print("Sample prompt:\n" + dataset[0]["prompt"])
    return dataset


prompt_dataset = build_prompt_dataset()


## Train

Initialise `GRPOTrainer` and call `.train()`.  
Expected time on A10G: ~2–3 hours for 150 steps.  
Watch the first 5 logging steps — reward should be non-zero and non-NaN.  
If reward is flat for 20+ steps, something is wrong — stop and debug on T4 first.

In [ ]:
# Uncomment after filling in Cell 10 (reward fn) and Cell 14 (dataset)

# prompt_dataset = build_prompt_dataset(n_samples=80)

# trainer = GRPOTrainer(
#     model=model,
#     reward_funcs=[driftenv_reward_fn],
#     args=config,
#     train_dataset=prompt_dataset,
#     tokenizer=tokenizer,
# )

# trainer.train()

## Save Trained LoRA Adapter

Save the adapter only — do NOT merge into the base model weights.  
Unsloth has a specific merge function (`model.save_pretrained_merged`) if you  
need a merged checkpoint later, but for the demo the adapter is sufficient.  
Push to HF Hub under your namespace so it's accessible for the eval cell.

In [ ]:
ADAPTER_DIR  = "driftenv_grpo_adapter"
HF_REPO_ID   = "harims95/driftenv-qwen1.5b-grpo"  # will be created if it doesn't exist

# model.save_pretrained(ADAPTER_DIR)
# tokenizer.save_pretrained(ADAPTER_DIR)
# print(f"Adapter saved locally to {ADAPTER_DIR}/")

# model.push_to_hub(HF_REPO_ID, token=True)
# tokenizer.push_to_hub(HF_REPO_ID, token=True)
# print(f"Adapter pushed to https://huggingface.co/{HF_REPO_ID}")

## Evaluation — Untrained vs Trained on Holdout

Run both model variants against the 5 holdout scenarios (IDs 1,3,7,14,20).  
Record all 4 reward components per step.  
Save to JSON — these become the before/after numbers in the README.

**Three numbers to report:**
1. Untrained Qwen 1.5B on holdout  
2. Trained Qwen 1.5B on holdout ← headline metric  
3. Qwen 72B reference on holdout: **~0.378** (already have this)

In [ ]:
import torch
import random as _random

# CRITICAL: use IDENTICAL generation kwargs for untrained AND trained models.
# Any difference in temperature, max_new_tokens, or sampling invalidates comparison.
GENERATION_KWARGS = dict(
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)


def _resolve_samples_dir() -> str:
    """Find the samples/ directory relative to wherever Colab is running."""
    candidates = [
        "samples",
        "../samples",
        "/content/driftenv/samples",
        "/kaggle/working/driftenv/samples",
        "/kaggle/working/samples",
    ]
    for path in candidates:
        parent = _os.path.dirname(_os.path.abspath(path))
        if _os.path.isdir(parent):
            _os.makedirs(path, exist_ok=True)
            return path
    # Last resort: create relative to cwd
    _os.makedirs("samples", exist_ok=True)
    return "samples"


def _derive_holdout_ids(dataset_path: str = None) -> set:
    """Read dataset.json and return the set of scenario IDs marked holdout=True."""
    if dataset_path is None:
        candidates = [
            "server/dataset.json",
            "../server/dataset.json",
            "/content/driftenv/server/dataset.json",
            "/kaggle/working/driftenv/server/dataset.json",
            "/kaggle/input/driftenv/server/dataset.json",
        ]
        for path in candidates:
            if _os.path.exists(path):
                dataset_path = path
                break
    if dataset_path is None:
        raise FileNotFoundError("dataset.json not found -- run Cell 14 first to locate it.")
    with open(dataset_path) as f:
        scenarios = _json.load(f)
    ids = {s["id"] for s in scenarios if s.get("holdout", False)}
    print(f"Holdout IDs derived from dataset.json: {sorted(ids)}")
    return ids


def _generate_completion(model, tokenizer, prompt: str) -> str:
    """Tokenize prompt, generate, decode -- returns only newly generated tokens."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_KWARGS,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_model(model, tokenizer, label: str, n_runs: int = 3) -> dict:
    """
    Evaluate model on all holdout scenarios, n_runs times each.

    Uses task="easy" (1-step) to match Cell 10 training episodes exactly.
    Iterates reset_env(holdout_only=True) until all holdout IDs collected per run.
    Saves results to samples/eval_{label}.json.

    Prerequisites:
      - Cell 14 must have run (populates _SCENARIO_LOOKUP and PROMPT_TEMPLATE)
      - Use same GENERATION_KWARGS for every evaluate_model call

    Returns summary dict: mean_total, mean_per_component, n_episodes, episodes.
    """
    if not _SCENARIO_LOOKUP:
        raise RuntimeError(
            "_SCENARIO_LOOKUP is empty. Run Cell 14 (build_prompt_dataset) first."
        )

    # Derive holdout IDs from dataset.json rather than hardcoding
    HOLDOUT_IDS = _derive_holdout_ids()

    # CRITICAL: Unsloth 2x inference speedup -- must call before model.generate
    FastLanguageModel.for_inference(model)
    _random.seed(42)

    all_episodes = []

    for run_idx in range(n_runs):
        seen_ids = set()
        attempts = 0

        while len(seen_ids) < len(HOLDOUT_IDS):
            attempts += 1
            if attempts > 100:
                print(
                    f"[eval WARNING] run {run_idx}: collected {seen_ids} after 100 attempts -- "
                    f"missing IDs: {HOLDOUT_IDS - seen_ids}"
                )
                break

            # task="easy" matches Cell 10 training (1-step episodes, fair comparison)
            obs_raw = reset_env(task="easy", holdout_only=True)
            obs = obs_raw.get("observation", obs_raw)
            instruction = obs.get("instruction", "")

            scenario_id = _SCENARIO_LOOKUP.get(instruction)
            if scenario_id is None:
                print(f"[eval WARNING] unknown instruction (not in lookup): '{instruction[:60]}'")
                continue
            if scenario_id in seen_ids:
                continue  # already collected this scenario this run

            seen_ids.add(scenario_id)

            prompt = PROMPT_TEMPLATE.format(instruction=instruction)
            completion = _generate_completion(model, tokenizer, prompt)

            result = step_env(action_text=completion)
            components = result.get("info", {}).get("rewards", {})
            total = float(result.get("reward", 0.0))

            all_episodes.append({
                "run":          run_idx,
                "scenario_id":  scenario_id,
                "prompt":       prompt,
                "response":     completion[:500],
                "components":   components,
                "total_reward": total,
            })

        print(
            f"[eval] run {run_idx+1}/{n_runs}: "
            f"{len(seen_ids)}/{len(HOLDOUT_IDS)} scenarios | {attempts} resets needed"
        )

    # Aggregate
    totals = [ep["total_reward"] for ep in all_episodes]
    mean_total = round(sum(totals) / len(totals), 4) if totals else 0.0

    component_keys = ["R_format", "R_interpretation", "R_pivot", "R_no_stale"]
    mean_per_component = {
        k: round(
            sum(ep["components"].get(k, 0.0) for ep in all_episodes) / len(all_episodes), 4
        ) if all_episodes else 0.0
        for k in component_keys
    }

    summary = {
        "label":              label,
        "mean_total":         mean_total,
        "mean_per_component": mean_per_component,
        "n_runs":             n_runs,
        "n_scenarios":        len(HOLDOUT_IDS),
        "n_episodes":         len(all_episodes),
        "generation_kwargs":  GENERATION_KWARGS,
        "episodes":           all_episodes,
    }

    samples_dir = _resolve_samples_dir()
    save_path = _os.path.join(samples_dir, f"eval_{label}.json")
    with open(save_path, "w") as f:
        _json.dump(summary, f, indent=2)

    # CRITICAL: switch back to training mode so subsequent trainer.train() works
    FastLanguageModel.for_training(model)

    print(f"\n[eval] === {label} ===")
    print(f"  mean_total:         {mean_total}")
    print(f"  mean_per_component: {mean_per_component}")
    print(f"  n_episodes:         {len(all_episodes)}")
    print(f"  saved:              {save_path}")
    return summary


# -- Usage: run these in order ------------------------------------------
#
# BEFORE training (get untrained baseline):
# untrained_results = evaluate_model(model, tokenizer, label="untrained_1.5B", n_runs=3)
#
# Run Cell 16 (trainer.train()) ...
#
# AFTER training:
# trained_results = evaluate_model(model, tokenizer, label="trained_1.5B", n_runs=3)
#
# Print the three-number comparison:
# print("\n=== FINAL COMPARISON (holdout set) ===")
# print(f"Qwen 72B  untrained (reference): 0.378")
# print(f"Qwen 1.5B untrained:             {untrained_results['mean_total']}")
# print(f"Qwen 1.5B trained (GRPO):        {trained_results['mean_total']}  <-- headline")


## Plotting

Two figures saved to `assets/` in the repo root:

1. **`reward_curves.png`** — 4 reward components over training steps (from `reward_log`)
2. **`before_after.png`** — grouped bar chart: untrained 1.5B vs trained 1.5B vs 72B reference, per difficulty tier

Commit both PNGs to `multi-reward-v2` and embed in README.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ASSETS_DIR = "../assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

def plot_reward_curves(log: list, save_path: str) -> None:
    """
    Plot the 4 reward component curves over training steps.
    log: list of dicts from reward_log (populated during training)
    """
    # TODO: fill in tomorrow after training completes
    # Skeleton:
    #
    # steps = [e["step"] for e in log]
    # fig, ax = plt.subplots(figsize=(10, 5))
    # for component, color in [("R_format", "grey"), ("R_interpretation", "blue"),
    #                           ("R_pivot", "green"), ("R_no_stale", "red")]:
    #     vals = [e.get(component, 0) for e in log]
    #     ax.plot(steps, vals, label=component, color=color, alpha=0.8)
    # ax.set_xlabel("Training step")
    # ax.set_ylabel("Reward component")
    # ax.set_title("DriftEnv — GRPO reward components over training")
    # ax.legend()
    # fig.tight_layout()
    # fig.savefig(save_path, dpi=150)
    # print(f"Saved: {save_path}")
    pass


def plot_before_after(untrained: dict, trained: dict, save_path: str) -> None:
    """
    Grouped bar chart: untrained 1.5B vs trained 1.5B vs 72B reference.
    One group per difficulty tier (easy / medium / hard).
    """
    # TODO: fill in tomorrow after eval_on_holdout runs
    # Skeleton:
    #
    # tiers = ["easy", "medium", "hard"]
    # ref_scores = {"easy": 0.307, "medium": 0.370, "hard": 0.611}  # 72B reference
    # x = np.arange(len(tiers))
    # width = 0.25
    # fig, ax = plt.subplots(figsize=(8, 5))
    # ax.bar(x - width, [untrained[t] for t in tiers], width, label="Untrained 1.5B", color="#d9534f")
    # ax.bar(x,         [trained[t]   for t in tiers], width, label="Trained 1.5B",   color="#5cb85c")
    # ax.bar(x + width, [ref_scores[t] for t in tiers], width, label="72B reference",  color="#5bc0de", alpha=0.7)
    # ax.set_xticks(x); ax.set_xticklabels(tiers)
    # ax.set_ylabel("Mean reward")
    # ax.set_title("DriftEnv — Before vs After GRPO training (holdout set)")
    # ax.legend()
    # fig.tight_layout()
    # fig.savefig(save_path, dpi=150)
    # print(f"Saved: {save_path}")
    pass


# plot_reward_curves(reward_log,   save_path=f"{ASSETS_DIR}/reward_curves.png")
# plot_before_after(untrained_results, trained_results,
#                   save_path=f"{ASSETS_DIR}/before_after.png")